In [1]:
import os
import json
from PIL import Image

import torch
import torch.utils.data as data
import torchvision.transforms.v2 as tfs
import torch.nn as nn
import torch.optim as optim
from tqdm.notebook import tqdm

In [2]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

True
NVIDIA GeForce RTX 4070 Laptop GPU
cuda


In [3]:
class AlphabetDataset(data.Dataset):
    def __init__(self, path, train=True, transform=None):
        self.path = os.path.join(path, "train" if train else "test")
        self.transform = transform

        with open(os.path.join(path, "format.json"), "r") as fp:
            self.format = json.load(fp)

        self.length = 0
        self.files = []
        self.targets = torch.eye(26)

        for _dir, _target in self.format.items():
            path = os.path.join(self.path, _dir)
            list_files = os.listdir(path)
            self.length += len(list_files)
            self.files.extend(map(lambda _x: (os.path.join(path, _x), _target), list_files))

    def __getitem__(self, index):
        path_file, target = self.files[index]
        t = self.targets[target]
        img = Image.open(path_file)
        if self.transform:
            img = self.transform(img).ravel().float() / 255.0
        #return img, t
        return img, target

    def __len__(self):
        return self.length

In [4]:
class AlphabetNN(nn.Module):
    def __init__(self, input_dim, num_hidden, output_dim):
        super().__init__()
        self.layer1 = nn.Linear(input_dim, num_hidden)
        self.layer2 = nn.Linear(num_hidden, output_dim)

    def forward(self, x):
        x = self.layer1(x)
        x = nn.functional.relu(x)
        x = self.layer2(x)
        return x

In [5]:
model = AlphabetNN(28*28, 256, 26).to(device)
to_tensor = tfs.ToImage()
d_train = AlphabetDataset("emnist_letters_folder", transform=to_tensor)
train_data = data.DataLoader(d_train, batch_size=32, shuffle=True)

optimizer = optim.Adam(params=model.parameters(), lr=0.001)
loss_function = nn.CrossEntropyLoss()

epochs = 10
model.train()


AlphabetNN(
  (layer1): Linear(in_features=784, out_features=256, bias=True)
  (layer2): Linear(in_features=256, out_features=26, bias=True)
)

In [6]:
for _e in range(epochs):
    loss_mean = 0 # Среднее значение функции потерь по эпохе
    lm_count = 0 # Текущее количество слагаемых
    train_tqdm = tqdm(train_data, leave=True)
    for i, (x_train, y_train) in enumerate(train_tqdm):
        x_train = x_train.to(device)
        y_train = y_train.to(device)
        predict = model(x_train)
        loss = loss_function(predict, y_train)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        lm_count += 1
        loss_mean = loss_mean + (loss.item() - loss_mean) / lm_count
        train_tqdm.set_description(f"Epoch {_e+1}/{epochs} loss={loss_mean:.3f}")

  0%|          | 0/3120 [00:00<?, ?it/s]

  0%|          | 0/3120 [00:00<?, ?it/s]

  0%|          | 0/3120 [00:00<?, ?it/s]

  0%|          | 0/3120 [00:00<?, ?it/s]

  0%|          | 0/3120 [00:00<?, ?it/s]

  0%|          | 0/3120 [00:00<?, ?it/s]

  0%|          | 0/3120 [00:00<?, ?it/s]

  0%|          | 0/3120 [00:00<?, ?it/s]

  0%|          | 0/3120 [00:00<?, ?it/s]

  0%|          | 0/3120 [00:00<?, ?it/s]

In [7]:
d_test = AlphabetDataset("emnist_letters_folder", train=False, transform=to_tensor)
test_data = data.DataLoader(d_test, batch_size=500, shuffle=False)
Q = 0

print("Размер обучающей выборки:", len(d_train))
print("Размер тестовой выборки:", len(d_test))

model.eval()

Размер обучающей выборки: 99840
Размер тестовой выборки: 24960


AlphabetNN(
  (layer1): Linear(in_features=784, out_features=256, bias=True)
  (layer2): Linear(in_features=256, out_features=26, bias=True)
)

In [8]:
for x_test, y_test in test_data:
    with torch.no_grad():
        x_test = x_test.to(device)
        y_test = y_test.to(device)
        p = model(x_test)
        p = torch.argmax(p, dim=1)
        y = y_test
        Q += torch.sum(p == y).item()

Q /= len(d_test)
print(Q)

0.8905849358974359


# Небольшое сравнение
### Архитектура нейросети
Входной слой содержит 784 нейрона, так как изображение имеет размер 28x28 пикселей.

Выходной слой содержит 26 нейронов — по одному на каждую букву латинского алфавита.

Функция активации - ReLU

### Параметры нейросети №1
Количество нейронов на промежуточном слое - 32

Количество эпох - 2

Примерная точность - 0,75

### Параметры нейросети №2
Количество нейронов на промежуточном слое - 32

Количество эпох - 5

Примерная точность - 0,80

### Параметры нейросети №3
Количество нейронов на промежуточном слое - 32

Количество эпох - 10

Примерная точность - 0,80

### Параметры нейросети №4
Количество нейронов на промежуточном слое - 64

Количество эпох - 2

Примерная точность - 0,81

### Параметры нейросети №5
Количество нейронов на промежуточном слое - 128

Количество эпох - 2

Примерная точность - 0,85

### Параметры нейросети №6
Количество нейронов на промежуточном слое - 256

Количество эпох - 2

Примерная точность - 0,87

### Параметры нейросети №7
Количество нейронов на промежуточном слое - 256

Количество эпох - 5

Примерная точность - 0,88

### Параметры нейросети №8
Количество нейронов на промежуточном слое - 256

Количество эпох - 10

Примерная точность - 0,89
